In [0]:
# 1. Install Kaggle
%pip install kaggle

# 2. Get Kaggle credentials from Databricks secret
import os

os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get(
    scope="kaggle",
    key="api_token"
)

# 3. Authenticate with Kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("Kaggle authentication successful")

print("Olist dataset downloaded")

In [0]:
# 5. create volume for files

spark.sql("CREATE CATALOG IF NOT EXISTS olist")
spark.sql("CREATE SCHEMA IF NOT EXISTS olist.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS olist.bronze.raw_data")

In [0]:
# 6. Download Olist dataset

volume_path = "/Volumes/olist/bronze/raw_data"

api.dataset_download_files(
    "olistbr/brazilian-ecommerce",
    path=volume_path,
    unzip=True
)

# 7. Load the data
tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dataframes = {}
for name, filename in tables.items():
    dataframes[name] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{volume_path}/{filename}")
    )